In [1]:
import pandas as pd
# matplotlib : la librairie de base pour tracer des graphiques
import matplotlib.pyplot as plt
# seaborn : rend les graphiques plus jolis et ajoute des fonctions statistiques
import seaborn as sns

# Affiche les graphiques directement dans le notebook (utile hors JupyterLab récent)
%matplotlib inline
sns.set_style("whitegrid")

df = pd.read_csv('michelin(in).csv')

## 29 lignes cassées

Même bug que pour `products(in).csv` : certaines villes contiennent une virgule (ex: `"Washington, D.C."`). Le fichier source a alors encadré **toute la ligne** de guillemets pour protéger cette virgule, mais de façon non standard — `pandas.read_csv()` s'y perd et remplit toutes les autres colonnes de `NaN`.

On reconnaît ces lignes cassées facilement : ce sont celles qui commencent par un guillemet `"`. On réutilise exactement la même stratégie que dans le premier cours :
1. Si la ligne commence et finit par `"`, on enlève ces guillemets et on remplace les `""` internes par `"`
2. On découpe avec le module `csv`, qui gère correctement les virgules protégées par des guillemets

Cette fois on récupère les noms de colonnes directement depuis l'en-tête du fichier plutôt que de les recopier à la main.

In [ ]:
import csv


def parser_ligne(ligne):
    """Répare et découpe une ligne du fichier CSV."""
    ligne = ligne.rstrip("\r\n")
    if ligne.startswith('"') and ligne.endswith('"'):
        # Ligne "cassée" : toute la ligne était entre guillemets
        contenu = ligne[1:-1].replace('""', '"')
        return next(csv.reader([contenu]))
    else:
        # Ligne normale : le module csv suffit
        return next(csv.reader([ligne]))


lignes_propres = []
# encoding="utf-8-sig" : ignore le caractère invisible BOM en début de fichier
# (pd.read_csv le fait automatiquement, mais pas open() avec "utf-8" tout court)
with open("michelin(in).csv", encoding="utf-8-sig") as f:
    entete = next(f).rstrip("\r\n").split(",")  # on récupère les noms de colonnes
    for ligne in f:
        lignes_propres.append(parser_ligne(ligne))

df = pd.DataFrame(lignes_propres, columns=entete)

print("Dimensions après réparation :", df.shape)
df.head()

In [ ]:
# Vérification : la ligne "minibar" est maintenant complète (plus aucune valeur manquante)
df[df["name"].str.contains("minibar", case=False, na=False)]

### Effet de bord : il faut reconvertir les colonnes numériques

Contrairement à `pd.read_csv()` qui devine les types automatiquement, notre lecture manuelle ligne par ligne donne **tout en texte**. `latitude` et `longitude` doivent donc être reconverties en nombres à virgule (`float`) — `year` et `stars` seront traités juste après, dans la cellule de conversion de types déjà présente plus bas.

In [ ]:
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)

df.dtypes

### Deuxième piège : les valeurs manquantes ne sont pas de vrais `NaN`

Avec `pd.read_csv()`, une case vide devient automatiquement `NaN`. Avec notre lecture manuelle, une case vide devient une **chaîne de caractères vide** `""` — et le fichier utilise aussi parfois littéralement le texte `"N/A"`. Ni l'un ni l'autre n'est reconnu par `.isna()` tant qu'on ne les convertit pas explicitement.

In [ ]:
# On remplace les "faux vides" par de vrais NaN, reconnus par pandas
df = df.replace(["", "N/A"], pd.NA)

print(df.isna().sum())

On convertit les types incorects

In [ ]:
df["year"] = df["year"].astype("Int64")
df["stars"] = df["stars"].astype("Int64")


df.dtypes
df.head(10)

In [ ]:
lignes_a_probleme = df.isna().any(axis=1).sum()
print(f"\nLignes affectées par le bug de parsing : {lignes_a_probleme}")

In [ ]:
def reparer_encodage(texte):
    try:
        return texte.encode("cp1252").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return texte  # déjà correct, on ne touche pas

df["city"] = df["city"].apply(lambda x: reparer_encodage(x) if pd.notna(x) else x)
df["name"] = df["name"].apply(lambda x: reparer_encodage(x) if pd.notna(x) else x)


df.head(10)

In [ ]:
df.isna().sum()                      # combien de NaN par colonne
df.isna().sum(axis=1).value_counts() # combien de lignes ont 0, 1, 2... valeurs manquantes à la fois


## doublons `name` + `city` + `year`

547 restaurants apparaissent deux fois pour la même année, avec un nombre d'étoiles qui diffère de 1 (ex: 1 puis 2) alors que **toutes les autres colonnes sont identiques** (localisation, cuisine, prix...). C'est donc bien le même restaurant dupliqué, pas deux établissements différents.

Stratégie : pour chaque groupe `(name, city, year)`, on garde uniquement la ligne avec le nombre d'étoiles **maximum** (la note la plus à jour), et on supprime le doublon inférieur.

In [ ]:
avant = len(df)

# On trie par étoiles décroissantes, puis on ne garde que la 1ère ligne de chaque groupe
# (name, city, year) -> c'est donc celle avec le plus d'étoiles qui est conservée
df = df.sort_values("stars", ascending=False).drop_duplicates(
    subset=["name", "city", "year"], keep="first"
)

# On remet les lignes dans l'ordre d'origine (le tri précédent a mélangé l'index)
df = df.sort_index().reset_index(drop=True)

print(f"Lignes avant : {avant}, lignes après : {len(df)} (supprimées : {avant - len(df)})")

# Vérification : il ne reste plus aucun groupe (name, city, year) en double
verif = df.duplicated(subset=["name", "city", "year"]).sum()
print(f"Doublons name+city+year restants : {verif}")

## Sauvegarder le résultat propre

On fige ce travail de nettoyage dans un nouveau fichier `michelin_clean.csv`, pour ne pas avoir à tout rejouer à chaque fois qu'on veut analyser ces données.

In [ ]:
df.to_csv("michelin_clean.csv", index=False)
print("Fichier sauvegardé :", len(df), "lignes,", len(df.columns), "colonnes")

In [ ]:
# Vérification : on recharge le fichier avec un simple pd.read_csv (plus besoin du parser manuel !)
# et on confirme que tout est toujours correct
df_verif = pd.read_csv("michelin_clean.csv")

print("Dimensions :", df_verif.shape)
print()
print(df_verif.dtypes)
print()
print("Valeurs manquantes :")
print(df_verif.isna().sum())
df_verif.head()

Le nombre de restaurants etoilés par région

In [ ]:
(
    df.groupby("region")
    .size()
    .sort_values(ascending=False)
    .head(24)
)


répartition entre prix et population

In [ ]:
(
    df.groupby("price")
    .size()
    .sort_index(ascending=False)
    .head(10)
)

In [ ]:
df.groupby("price")["stars"].mean()


In [ ]:
df.groupby("price")["stars"].median()

Il n'y a pas de lien entre le prix et les etoiles

In [ ]:
prix_int = df["price"].str.len()

df.groupby(prix_int)["stars"].median().sort_index(ascending=False).head(10)

In [ ]:
pd.crosstab(df["price"], df["stars"])